# NB11 — Ridge Regression (L2 Regularisation)

> **Shrink large coefficients to reduce overfitting and handle multicollinearity.**

OLS minimises:  `SSR = Σ(yᵢ − ŷᵢ)²`

Ridge minimises: `SSR + λ·Σβⱼ²`

The penalty `λ·Σβⱼ²` (L2 norm) discourages large coefficients.
λ = 0 → OLS.   λ → ∞ → all coefficients → 0.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 50
X = np.sort(np.random.uniform(-2, 2, n))
y = np.sin(X * np.pi) + np.random.normal(0, 0.3, n)

X_2d = X.reshape(-1,1)
X_tr, X_te, y_tr, y_te = train_test_split(X_2d, y, test_size=0.3, random_state=0)
X_plot = np.linspace(-2, 2, 200).reshape(-1,1)

alphas = [0, 0.01, 1, 100]   # lambda values
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, alpha in zip(axes, alphas):
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=10, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=alpha) if alpha > 0 else LinearRegression()),
    ])
    pipe.fit(X_tr, y_tr)
    tr_r2 = pipe.score(X_tr, y_tr)
    te_r2 = pipe.score(X_te, y_te)
    ax.scatter(X, y, s=15, alpha=0.5, color='steelblue')
    ax.plot(X_plot, pipe.predict(X_plot), 'r-', linewidth=2)
    label = f'λ={alpha}' if alpha > 0 else 'OLS (λ=0)'
    ax.set_title(f'{label}\nTrain R²={tr_r2:.3f}\nTest R²={te_r2:.3f}')
    ax.set_ylim(-2, 2); ax.grid(alpha=0.3)

plt.suptitle('Ridge: higher λ → smoother fit (less overfitting)', fontsize=12)
plt.tight_layout(); plt.show()


## Ridge closed-form solution

```
β̂_ridge = (XᵀX + λI)⁻¹ Xᵀy
```

The `λI` term makes the matrix invertible even when XᵀX is near-singular (multicollinearity).
This is why Ridge is the standard fix for multicollinearity.


In [ ]:
# Implement Ridge from scratch using normal equations
import numpy as np

def ridge_fit(X_design, y, lam):
    n, p = X_design.shape
    # Don't penalise the intercept column (column 0)
    I_pen = np.eye(p)
    I_pen[0, 0] = 0
    beta = np.linalg.solve(X_design.T @ X_design + lam * I_pen,
                           X_design.T @ y)
    return beta

np.random.seed(1)
X_raw = np.random.randn(100, 3)
y_true = 2*X_raw[:,0] - X_raw[:,1] + 0.5*X_raw[:,2]
y = y_true + np.random.normal(0, 0.5, 100)

X_d = np.column_stack([np.ones(100), X_raw])

for lam in [0, 0.1, 1, 10, 100]:
    b = ridge_fit(X_d, y, lam)
    print(f"λ={lam:>5}: β = [{b[1]:.3f}, {b[2]:.3f}, {b[3]:.3f}]  (true: [2, -1, 0.5])")


In [ ]:
# Regularisation path: how coefficients shrink as λ increases
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X_std = StandardScaler().fit_transform(housing.data)
y     = housing.target

lambdas = np.logspace(-3, 5, 200)
coefs   = []

for lam in lambdas:
    from sklearn.linear_model import Ridge
    m = Ridge(alpha=lam).fit(X_std, y)
    coefs.append(m.coef_)

coefs = np.array(coefs)

plt.figure(figsize=(10, 5))
for j, name in enumerate(housing.feature_names):
    plt.semilogx(lambdas, coefs[:, j], label=name)
plt.xlabel('λ (log scale)'); plt.ylabel('Coefficient value')
plt.title('Ridge regularisation path — all coefficients shrink to 0 as λ→∞')
plt.legend(fontsize=8, loc='upper right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Find optimal λ via cross-validation
rc = RidgeCV(alphas=lambdas, cv=5).fit(X_std, y)
print(f"Optimal λ (RidgeCV): {rc.alpha_:.4f}")


## Key Takeaways

| Property | Ridge |
|---------|-------|
| Penalty | λ·Σβⱼ² (L2 norm) |
| Effect | Shrinks all coefficients toward 0 |
| Variable selection | No — keeps all features, just smaller |
| Best for | Multicollinearity, when all features contribute |
| Closed form | Yes: β̂ = (XᵀX + λI)⁻¹Xᵀy |

**Next:** NB12 — Lasso (L1) which performs variable selection.
